# 🏙️ Dubai Real Estate – End to End ML Project
**Data:** Dubai Land Department transactions (Jan – Nov 2026)

**Is project mein 3 kaam hain:**
1. **Regression** – property ka *price per sq.m* predict karna (Linear Regression, Decision Tree, Random Forest)
2. **Classification** – deal *"Good Deal"* (peers se sasti) hai ya nahi (Logistic Regression, Decision Tree, Random Forest) → Accuracy, Precision, Recall
3. **Clustering** – areas ko groups mein todna + investment score (K-Means)

Code easy hai: pandas + numpy + seaborn, bina for-loop ke. Har line ke baad Hinglish comment hai.

## 1. Imports 

In [3]:
import numpy as np                                                    # numbers aur maths ke liye
import pandas as pd                                                   # data table (DataFrame) ke liye
import matplotlib.pyplot as plt                                       # charts ka base library
import seaborn as sns                                                 # sundar charts ke liye- ke liye
import joblib, os                                                     # model save karne aur folder banane ke liye

from sklearn.model_selection import RandomizedSearchCV                # best settings dhoondhne ke liye
from sklearn.linear_model import LinearRegression, LogisticRegression # linear models
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier # decision tree models
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier # random forest models
from sklearn.cluster import KMeans                                    # clustering model
from sklearn.metrics import (r2_score, accuracy_score, precision_score,recall_score, f1_score, classification_report, confusion_matrix, silhouette_score)

In [4]:
sns.set_theme(style="whitegrid", palette="deep")                      # seaborn ka default look set karo
pd.set_option("display.width", 200)                                   # table ki width badhao
pd.set_option("display.max_columns", 30)                              # zyada columns dikhao
os.makedirs("models", exist_ok=True)                                  # models ke liye folder banao

## Data load

In [5]:
df = pd.read_csv("Dubai_Realstate_transactions-2026-11-07.csv")       # CSV file padho

In [6]:
df.columns

Index(['TRANSACTION_NUMBER', 'INSTANCE_DATE', 'GROUP_EN', 'PROCEDURE_EN', 'IS_OFFPLAN_EN', 'IS_FREE_HOLD_EN', 'USAGE_EN', 'AREA_EN', 'PROP_TYPE_EN', 'PROP_SB_TYPE_EN', 'TRANS_VALUE',
       'PROCEDURE_AREA', 'ACTUAL_AREA', 'ROOMS_EN', 'PARKING', 'NEAREST_METRO_EN', 'NEAREST_MALL_EN', 'NEAREST_LANDMARK_EN', 'TOTAL_BUYER', 'TOTAL_SELLER', 'MASTER_PROJECT_EN', 'PROJECT_EN'],
      dtype='str')

In [7]:
df["date"] = pd.to_datetime(df["INSTANCE_DATE"], format="%d-%m-%Y %H:%M") # date text ko asli date banao

In [8]:
df.shape

(186662, 23)

In [9]:
df.head()

,TRANSACTION_NUMBER,INSTANCE_DATE,GROUP_EN,PROCEDURE_EN,IS_OFFPLAN_EN,IS_FREE_HOLD_EN,USAGE_EN,AREA_EN,PROP_TYPE_EN,PROP_SB_TYPE_EN,TRANS_VALUE,PROCEDURE_AREA,ACTUAL_AREA,ROOMS_EN,PARKING,NEAREST_METRO_EN,NEAREST_MALL_EN,NEAREST_LANDMARK_EN,TOTAL_BUYER,TOTAL_SELLER,MASTER_PROJECT_EN,PROJECT_EN,date
0,101-10-2026,01-07-2026 16:00,Mortgage,Portfolio Mortgage Registration Pre-Registration,Off-Plan,Free Hold,Residential,BUSINESS BAY,Unit,Flat,1699451.22,99.32,99.32,1 B/R,1,Buj Khalifa Dubai Mall Metro Station,Dubai Mall,Downtown Dubai,0,0,NaN,THE CRESTMARK,2026-07-01 16:00:00
1,101-10-2026,01-07-2026 16:00,Mortgage,Portfolio Mortgage Registration Pre-Registration,Off-Plan,Free Hold,Residential,BUSINESS BAY,Unit,Flat,1699451.22,46.17,46.17,Studio,1,Buj Khalifa Dubai Mall Metro Station,Dubai Mall,Downtown Dubai,0,0,NaN,THE CRESTMARK,2026-07-01 16:00:00
2,101-10-2026,01-07-2026 16:00,Mortgage,Portfolio Mortgage Registration Pre-Registration,Off-Plan,Free Hold,Residential,BUSINESS BAY,Unit,Flat,1699451.22,46.15,46.15,Studio,1,Buj Khalifa Dubai Mall Metro Station,Dubai Mall,Downtown Dubai,0,0,NaN,THE CRESTMARK,2026-07-01 16:00:00
3,101-10-2026,01-07-2026 16:00,Mortgage,Portfolio Mortgage Registration Pre-Registration,Off-Plan,Free Hold,Residential,BUSINESS BAY,Unit,Flat,1699451.22,91.81,91.81,1 B/R,1,Buj Khalifa Dubai Mall Metro Station,Dubai Mall,Downtown Dubai,0,0,NaN,THE CRESTMARK,2026-07-01 16:00:00
4,101-11-2026,01-07-2026 16:29,Mortgage,Portfolio Mortgage Registration Pre-Registration,Off-Plan,Free Hold,Residential,Palm Deira,Unit,Shop,6608107.80,417.91,417.91,NaN,6,NaN,NaN,NaN,0,0,NaN,ELLINGTON SANDS I,2026-07-01 16:29:00


## 2. Data Cleaning
- Sirf **asli buy/sell deals** (mortgage, gift, grant nahi)
- Sirf **residential Flat / Villa / Hotel Apartment**
- Galat prices (typo / bulk deals) hata do: price per sq.m ka bottom 1% aur top 1%

In [10]:
df = df[df["PROCEDURE_EN"].isin(["Sale", "Sell - Pre registration", "Delayed Sell"])]  # sirf buy/sell deals rakho
df = df[df["PROP_SB_TYPE_EN"].isin(["Flat", "Villa", "Hotel Apartment"]) & (df["USAGE_EN"] == "Residential")]  # sirf residential
df = df[df["ACTUAL_AREA"].between(15, 1000) & (df["TRANS_VALUE"] > 0)]  # bahut chhoti/badi area aur zero price hatao
df = df.drop_duplicates("TRANSACTION_NUMBER")                         # duplicate transactions hatao

In [11]:
df["ppsm"] = df["TRANS_VALUE"] / df["ACTUAL_AREA"]                    # price per sq.m banao (yahi hamara target hai)
low, high = df["ppsm"].quantile([0.01, 0.99])                         # bottom 1% aur top 1% ki limit nikalo
df = df[df["ppsm"].between(low, high)].copy()                         # extreme prices hata do
print("Clean rows:", len(df))                                         # kitni rows bachi dikhao
df["ppsm"].describe().round(0)                                        # price per sq.m ka summary

Clean rows: 114069


count    114069.0
mean      20134.0
std        7382.0
min        6176.0
25%       15232.0
50%       18555.0
75%       23424.0
max       48383.0
Name: ppsm, dtype: float64

## 3. Feature Engineering

Naye columns aur missing values

In [22]:
df["log_area"]    = np.log(df["ACTUAL_AREA"])                         # size ka log (skew kam karne ke liye)
df["month_idx"]   = (df["date"] - pd.Timestamp("2026-01-01")).dt.days / 30.4  # kitne mahine beete (time feature)
df["has_parking"] = df["PARKING"].notna().astype(int)                 # parking hai to 1, nahi to 0
df["month"]       = df["date"].dt.to_period("M").astype(str)          # month ka naam (chart ke liye)

df["ROOMS_EN"]   = df["ROOMS_EN"].fillna("Unknown")                   # missing rooms ko 'Unknown' bharo
df["PROJECT_EN"] = df["PROJECT_EN"].fillna("Unknown")                 # missing project ko 'Unknown' bharo
near = ["NEAREST_METRO_EN", "NEAREST_MALL_EN", "NEAREST_LANDMARK_EN"] # nearby places ke columns
df[near] = df[near].fillna("None")                                    # missing nearby ko 'None' bharo

Cell 2 – Rare projects aur peer features

In [26]:
project_count = df["PROJECT_EN"].map(df["PROJECT_EN"].value_counts()) # har row ke project ki sales count
df["PROJECT_EN"] = df["PROJECT_EN"].where(project_count >= 50, "Other") # chhote projects ko 'Other' bana do

peer = ["AREA_EN", "PROP_SB_TYPE_EN", "ROOMS_EN"]                     # peer group = same area + type + rooms
peer_area = df.groupby(peer)["ACTUAL_AREA"].transform("median")       # peer group ki median size
df["size_vs_peer"] = (df["ACTUAL_AREA"] / peer_area).clip(0, 5)       # unit peers se kitni badi/chhoti hai
proj_area = df.groupby("PROJECT_EN")["ACTUAL_AREA"].transform("median") # project ki median size
df["size_vs_project"] = (df["ACTUAL_AREA"] / proj_area).clip(0, 5)    # unit project mein kitni badi/chhoti hai
df["peer_count"] = np.log(df.groupby(peer)["ppsm"].transform("size")) # peer group mein kitni deals (popularity)

In [ ]:
cat_Cols = ['AREA_EN','PROJECT_EN','PROP_SB_TYPE_EN','ROOMS_EN','IS_OFFPLAN_EN','IS_FREE_HOLD_EN','NEAREST_METRO_EN','NEAREST_MALL_EN','NEAREST_LANDMARK_EN']
num_cols = ['log_area','month_idx','has_parking','size_vs_peer','size_vsyu']

In [ ]:
cat_cols = ["AREA_EN", "PROJECT_EN", "PROP_SB_TYPE_EN", "ROOMS_EN", "IS_OFFPLAN_EN",
            "IS_FREE_HOLD_EN", "NEAREST_METRO_EN", "NEAREST_MALL_EN", "NEAREST_LANDMARK_EN"]  # text wale columns
num_cols = ["log_area", "month_idx", "has_parking", "size_vs_peer", "size_vs_project", "peer_count"]  # number wale columns

X_tree = df[cat_cols].apply(lambda s: s.astype("category").cat.codes) # trees ke liye: text ko numbers (codes) mein badlo
X_tree[num_cols] = df[num_cols]                                       # number columns saath jodo

X_lin = pd.get_dummies(df[cat_cols], dtype="float32")                 # linear models ke liye: one-hot encoding
X_lin[num_cols] = ((df[num_cols] - df[num_cols].mean()) / df[num_cols].std()).astype("float32")  # numbers ko scale karo

print("Tree features:", X_tree.shape, "| Linear features:", X_lin.shape)  # dono tables ka size

Tree features: (114069, 15) | Linear features: (114069, 688)
